# Supervised Streaming Structural-Break Detector

A supervised upgrade over the EWMA baseline. At each online step we compute a small vector of
**two-sample features** comparing a trailing online window against the historical segment, and a
gradient-boosted classifier (`HistGradientBoostingClassifier`) maps that vector to a break
probability.

**Features (window vs history):** mean z-score, variance log-ratio, KS statistic, skew/kurtosis
differences, autocorrelation lag-1/2 differences, IQR ratio, tail masses (>2σ, >3σ), zero-crossing
rate, normalized slope, spectral-centroid difference, range ratio. These reuse last year's
distribution-comparison ideas (which scored LB 88.60% on the one-shot task), adapted to the
streaming per-step TS-AUC protocol.

**Budget control:** features + prediction are recomputed every `PREDICT_EVERY` steps and held
constant in between; the window is maintained in O(1).

Local validation (800 held-out train series): **TS-AUC ≈ 0.57**, versus 0.516 for the EWMA baseline.

## Getting set up

Install the Crunch CLI and initialize the local environment (downloads the data and wires up the
tooling). Get your own token from the competition page and paste it below.

>> https://hub.crunchdao.com/competitions/structural-break-real-time/submit/notebook

In [ ]:
%pip install crunch-cli --upgrade --quiet --progress-bar off
!crunch setup-notebook structural-break-real-time vH28wOW4maIQ1xYqgTEfGm6f

In [ ]:
import os

# Keep BLAS / OpenMP single-threaded. infer() runs in several parallel
# processes, and single-row model predictions are dramatically faster without
# per-call thread-pool spin-up. Env vars only work if set before numpy loads
# (not guaranteed on the cloud), so infer() ALSO caps threads at call time via
# threadpool_limits below.
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")

import math
from collections import deque
from typing import Iterable, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

In [ ]:
import crunch

# Load the Crunch Toolings (data loader, local tester, submitter).
crunch_tools = crunch.load_notebook()

In [ ]:
# Load the data (only needed for local exploration / testing).
train_data, test_data = crunch_tools.load_data()
print(f"train series: {len(train_data)}  |  test series: {len(test_data)}")

## Feature extractor

Shared by `train()` and `infer()` so the training features and the streaming features are computed
identically. `hist_context()` summarizes the historical segment once per series; `compute_features()`
turns a trailing online window into the 15-dim feature vector.

In [ ]:
FEATURE_NAMES = [
    "mean_z", "log_var_ratio", "ks", "skew_diff", "kurt_diff",
    "ac1_diff", "ac2_diff", "iqr_ratio", "tail2", "tail3",
    "zero_cross", "slope_norm", "spec_centroid_diff", "rng_ratio",
    "spec_entropy",
]


def _autocorr(a, lag):
    if len(a) <= lag:
        return 0.0
    x = a[:-lag]
    y = a[lag:]
    xm = x.mean()
    ym = y.mean()
    denom = math.sqrt(((x - xm) ** 2).sum() * ((y - ym) ** 2).sum())
    if denom < 1e-12:
        return 0.0
    return float(((x - xm) * (y - ym)).sum() / denom)


def _spectral_centroid(a):
    a = a - a.mean()
    sp = np.abs(np.fft.rfft(a))
    freqs = np.fft.rfftfreq(len(a))
    s = sp.sum()
    if s < 1e-12:
        return 0.0
    return float((freqs * sp).sum() / s)


def _skew_kurt(w, mean_w, pop_var):
    """Population skew and excess kurtosis from central moments (pure numpy)."""
    if pop_var < 1e-12:
        return 0.0, 0.0
    d = w - mean_w
    m3 = float(np.mean(d ** 3))
    m4 = float(np.mean(d ** 4))
    sk = m3 / (pop_var ** 1.5)
    ku = m4 / (pop_var ** 2) - 3.0
    return sk, ku


def _fast_ks(w_sorted, h_sorted):
    """Two-sample KS statistic from pre-sorted arrays (numpy searchsorted)."""
    h = len(h_sorted)
    n = len(w_sorted)
    allv = np.concatenate([h_sorted, w_sorted])
    cdf_h = np.searchsorted(h_sorted, allv, side="right") / h
    cdf_w = np.searchsorted(w_sorted, allv, side="right") / n
    return float(np.max(np.abs(cdf_h - cdf_w)))


def hist_context(x_h, hist_cap):
    """Precompute historical-segment statistics + a sorted sample for the KS test."""
    n = len(x_h)
    mu = float(x_h.mean()) if n else 0.0
    var = float(x_h.var(ddof=1)) if n > 1 else 1.0
    var = max(var, 1e-12)
    sd = math.sqrt(var)
    pop_var = float(x_h.var()) if n else 1.0
    sk, ku = _skew_kurt(x_h, mu, pop_var) if n > 2 else (0.0, 0.0)
    ac1 = _autocorr(x_h, 1)
    ac2 = _autocorr(x_h, 2)
    q25, q75 = np.quantile(x_h, [0.25, 0.75]) if n else (0.0, 0.0)
    iqr = float(q75 - q25)
    iqr = iqr if iqr > 1e-12 else 1e-12
    rng = float(x_h.max() - x_h.min()) if n else 1.0
    rng = rng if rng > 1e-12 else 1e-12
    sc = _spectral_centroid(x_h) if n > 4 else 0.0
    if n > hist_cap:
        stride = n // hist_cap
        sample = x_h[::stride][:hist_cap]
    else:
        sample = x_h
    sample_sorted = np.sort(np.asarray(sample, dtype=np.float64))
    return {
        "mu": mu, "sd": sd, "var": var, "skew": sk, "kurt": ku,
        "ac1": ac1, "ac2": ac2, "iqr": iqr, "rng": rng, "sc": sc,
        "sample_sorted": sample_sorted,
    }


def compute_features(w, ctx):
    """w: 1-D np.array trailing window. ctx: hist_context dict. -> feature list (len 15).

    Pure-numpy (no scipy.stats) so it is fast enough for per-step streaming inference.
    """
    n = len(w)
    mean_w = float(w.mean())
    var_w = max(float(w.var()), 1e-12)
    se = ctx["sd"] / math.sqrt(n)
    mean_z = (mean_w - ctx["mu"]) / max(se, 1e-8)
    log_var_ratio = math.log(var_w / ctx["var"])
    w_sorted = np.sort(w)
    ks = _fast_ks(w_sorted, ctx["sample_sorted"])
    sk, ku = _skew_kurt(w, mean_w, var_w)
    skew_diff = sk - ctx["skew"]
    kurt_diff = ku - ctx["kurt"]
    ac1_diff = _autocorr(w, 1) - ctx["ac1"]
    ac2_diff = _autocorr(w, 2) - ctx["ac2"]
    q25 = w_sorted[int(0.25 * (n - 1))]
    q75 = w_sorted[int(0.75 * (n - 1))]
    iqr_w = max(float(q75 - q25), 1e-12)
    iqr_ratio = iqr_w / ctx["iqr"]
    dev = np.abs(w - ctx["mu"]) / ctx["sd"]
    tail2 = float((dev > 2.0).mean())
    tail3 = float((dev > 3.0).mean())
    centered = w - ctx["mu"]
    zero_cross = float((np.diff(np.sign(centered)) != 0).mean()) if n > 1 else 0.0
    i = np.arange(n, dtype=np.float64)
    sx = i.sum()
    sxx = (i * i).sum()
    sy = w.sum()
    sxy = (i * w).sum()
    denom = n * sxx - sx * sx
    slope = (n * sxy - sx * sy) / denom if abs(denom) > 1e-12 else 0.0
    slope_norm = slope * n / ctx["sd"]
    spec_centroid_diff = _spectral_centroid(w) - ctx["sc"]
    rng_ratio = float(w_sorted[-1] - w_sorted[0]) / ctx["rng"]
    # spectral entropy of the window (E13: robust +0.0050 TS-AUC @ WINDOW=130, t=+4.67)
    ps = np.abs(np.fft.rfft(w - mean_w)) ** 2
    ps_sum = ps.sum()
    if ps_sum < 1e-12 or len(ps) < 2:
        spec_entropy = 0.0
    else:
        p = ps / ps_sum
        p = p[p > 0]
        spec_entropy = float(-(p * np.log(p)).sum()) / math.log(len(ps))
    return [
        mean_z, log_var_ratio, ks, skew_diff, kurt_diff,
        ac1_diff, ac2_diff, iqr_ratio, tail2, tail3,
        zero_cross, slope_norm, spec_centroid_diff, rng_ratio,
        spec_entropy,
    ]


def compute_features_ms(w, ctx):
    """Multi-scale wrapper (E17 winner ms_60_130_250, 45 feats, 10-fold t=+3.06).

    Compute the base15 feature set on nested trailing sub-windows w[-s:] for each
    scale s in MS_SCALES and concatenate. Short scales are responsive; long scales
    are slow -> the model sees several views of the same series-vs-history contrast.
    Guarded so a scale larger than the current window just reuses the full window.
    """
    n = len(w)
    out = []
    for s in MS_SCALES:
        ws = w if s >= n else w[-s:]
        out.extend(compute_features(ws, ctx))
    return out


## Configuration

`WINDOW`, `MIN_N`, `HIST_CAP` shape the features; `PREDICT_EVERY` throttles inference cost;
`STEPS_PER_SERIES` / `SEED` control training-set sampling.

In [ ]:
# @crunch/keep:on
WINDOW           = 250   # buffer = largest MS scale (E17 multi-scale)
MS_SCALES        = (60, 130, 250)  # E17 champion ms_60_130_250 (t=+3.06, LB 57.50%/214); (40,..)=57.38%/216
MIN_N            = 15    # warm-up: emit 0.0 until this many online points seen
HIST_CAP         = 200   # max historical points sampled for the KS test
PREDICT_EVERY    = 1     # recompute features + prediction every N steps (1 = no staleness)
STEPS_PER_SERIES = 10    # sampled online steps per training series
MAX_TRAIN_SERIES = 10**9 # consume ALL series the platform provides (no cap)
MAX_ITER         = 200   # gradient-boosting trees (fewer -> faster per-step predict)
SEED             = 42
ENSEMBLE_SEEDS   = (42, 7, 123, 2024, 31337)  # seed-ensemble: avg proba over K champions (variance reduction)

## `train()`

For each training series we compute `hist_context` once, sample `STEPS_PER_SERIES` online positions,
build the 15-dim feature vector for the trailing window at each, and label it `1` if that position is
at/after the break `tau` (else `0`). A `HistGradientBoostingClassifier` is fit on the pooled rows and
saved to disk.

In [ ]:
def train(
    datasets: List[Tuple[int, List[float], List[float], Optional[int]]],
    model_directory_path: str,
):
    rng = np.random.default_rng(SEED)
    rows: List[List[float]] = []
    labels: List[int] = []

    n_used = 0
    for dataset_id, x_hist, x_online, tau in datasets:
        if n_used >= MAX_TRAIN_SERIES:
            break
        x_h = np.asarray(x_hist, dtype=np.float64)
        x_on = np.asarray(list(x_online), dtype=np.float64)  # materialize (iterable once)
        L = len(x_on)
        if L <= MIN_N or len(x_h) < 5:
            continue
        ctx = hist_context(x_h, HIST_CAP)
        cand = np.arange(MIN_N, L)
        k = min(STEPS_PER_SERIES, len(cand))
        picks = rng.choice(cand, size=k, replace=False)
        for t in picks:
            lo = max(0, t - WINDOW + 1)
            w = x_on[lo:t + 1]
            rows.append(compute_features_ms(w, ctx))
            labels.append(1 if (tau is not None and t >= tau) else 0)
        n_used += 1

    X_train = np.asarray(rows, dtype=np.float64)
    y_train = np.asarray(labels, dtype=np.int64)
    print(f"train matrix {X_train.shape} from {n_used} series, positives {y_train.mean():.3f}")

    models = []
    for _s in ENSEMBLE_SEEDS:
        _m = HistGradientBoostingClassifier(
            max_iter=MAX_ITER, learning_rate=0.05, max_leaf_nodes=31,
            l2_regularization=1.0, early_stopping=True, random_state=_s,
        )
        _m.fit(X_train, y_train)
        models.append(_m)
    print(f"trained {len(models)} seed-ensemble members ({models[0].n_iter_} trees in member 0)")

    joblib.dump(
        {
            "models": models,
            "config": {
                "WINDOW": WINDOW,
                "MS_SCALES": list(MS_SCALES),
                "MIN_N": MIN_N,
                "HIST_CAP": HIST_CAP,
                "PREDICT_EVERY": PREDICT_EVERY,
            },
        },
        os.path.join(model_directory_path, "model.joblib"),
    )

## `infer()`

A generator yielding exactly one score per online point. Per series we compute `hist_context` once,
then maintain a trailing window (O(1) add/evict). Every `PREDICT_EVERY` steps we recompute the
feature vector and the model's break probability; between refreshes the last score is held. Points
before `MIN_N` yield `0.0`.

In [ ]:
def infer(
    datasets: Iterable[Tuple[List[float], Iterable[float]]],
    model_directory_path: str,
):
    saved = joblib.load(os.path.join(model_directory_path, "model.joblib"))
    models = saved["models"]
    cfg = saved["config"]
    window        = int(cfg["WINDOW"])
    min_n         = int(cfg["MIN_N"])
    hist_cap      = int(cfg["HIST_CAP"])
    predict_every = int(cfg["PREDICT_EVERY"])
    global MS_SCALES
    MS_SCALES = tuple(cfg.get("MS_SCALES", MS_SCALES))

    yield  # Signal readiness to the runner.

    # Force single-threaded BLAS/OpenMP at call time. Setting env vars is not
    # enough on the cloud: numpy/sklearn are imported before this notebook runs,
    # and infer workers are separate processes. Without this cap, N parallel
    # workers each spawn many OpenMP threads, oversubscribing the CPUs and making
    # every single-row predict_proba orders of magnitude slower.
    with threadpool_limits(limits=1):
        for x_historical, x_online in datasets:
            x_h = np.asarray(x_historical, dtype=np.float64)
            ctx = hist_context(x_h, hist_cap)

            buf = deque()
            score = 0.0
            for idx, point in enumerate(x_online):
                if len(buf) == window:
                    buf.popleft()
                buf.append(float(point))
                n = len(buf)

                if n < min_n:
                    yield 0.0
                    continue

                if idx % predict_every == 0:
                    w = np.asarray(buf)
                    feats = np.asarray(compute_features_ms(w, ctx), dtype=np.float64).reshape(1, -1)
                    score = float(np.mean([m.predict_proba(feats)[0, 1] for m in models]))

                yield float(score)

In [ ]:
# @crunch/keep:on
# Each worker is pinned to 1 BLAS/OpenMP thread (see infer), so we can safely
# use most of the available vCPUs. Auto-scales to whatever machine you pick.
INFER_PARALLELISM = max(1, (os.cpu_count() or 4) - 1)
# INFER_PARALLELISM = 1  # set to 1 while debugging

## Local test + TS-AUC

Runs the same flow the platform will run, then computes the leaderboard metric locally.

In [ ]:
crunch_tools.test(
    # force_first_train=False,
    # no_determinism_check=True,
)

In [ ]:
# Compute local TS-AUC (same as the leaderboard, on the local test split).
prediction = pd.read_parquet("prediction/prediction.parquet")
y_test = pd.read_parquet("data/y_test.reduced.parquet")

merged = prediction.merge(y_test, how="left", left_index=True, right_index=True)
merged["time_online"] = merged.groupby("id").cumcount()

weighted_auc_sum = 0.0
total_weight = 0.0
for t, group in merged.groupby("time_online"):
    labels = group["target"].values
    scores = group["prediction"].values
    n_pos = int(labels.sum())
    n_neg = int((1 - labels).sum())
    if n_pos == 0 or n_neg == 0:
        continue
    auc_t = float(roc_auc_score(labels, scores))
    weight = float(n_pos * n_neg)
    weighted_auc_sum += weight * auc_t
    total_weight += weight

ts_auc = weighted_auc_sum / total_weight if total_weight > 0 else 0.5
print(f"Local TS-AUC: {ts_auc:.4f}")

## Submit

On Colab, run the cell below. Locally, download this notebook and upload it to the platform, then create a run.

In [ ]:
Message = ""
crunch_tools.submit(message=Message)